# 第 6 章 · 图谱多跳与路径排序

鲁迅/莫言图谱：带关系类型的多跳 JOIN 与软路径计票。

配套交互演示：[章节网页](../ch6.html)

## 本节目标

- 理解三元组存储
- 多跳查询逐步执行
- 关系约束为何必要
- 软路径计票

## 1. 三元组与查询模板

知识图谱存 **(头实体, 关系, 尾实体)**。查询模板约束关系类型，避免语义错误的边。

**模板**

```text
(鲁迅, 创作, ?X) AND (?X, 发表于, ?Y)
```

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "labs").exists() and (ROOT.parent / "labs").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "labs"))
sys.path.insert(0, str(ROOT / "labs" / "ch06"))
import matplotlib.pyplot as plt
from common.mpl_setup import configure_matplotlib
configure_matplotlib()
from IPython.display import display, Image
from reasoning import *

In [2]:
kg = load_kg()

In [3]:
print('pattern', kg['query']['pattern'])

pattern ['鲁迅', '创作', '?X', '?X', '发表于', '?Y']


In [4]:
print('Y =', kg['query']['answer_y'])

Y = 文学周报社


## 2. 边列表与邻接

In [5]:
for h,r,t in kg['edges']: print(f'({h})-[{r}]->({t})')

(鲁迅)-[创作]->(狂人日记)
(鲁迅)-[创作]->(呐喊)
(狂人日记)-[发表于]->(文学周报社)
(呐喊)-[发表于]->(文学周报社)
(狂人日记)-[获得]->(茅盾文学奖)
(莫言)-[创作]->(蛙)
(莫言)-[创作]->(红高粱)
(蛙)-[获得]->(茅盾文学奖)
(红高粱)-[入选]->(典藏)
(红高粱)-[改编]->(电影)
(电影)-[获得]->(金熊奖)


In [6]:
adj = build_adj(kg)

In [7]:
print('鲁迅 出边:', adj['鲁迅'])

鲁迅 出边: [('创作', '狂人日记'), ('创作', '呐喊')]


**思考** · 「发表于」边连接什么？

<details><summary>查看答案</summary>

作品 → 刊物/地点。

</details>

## 3. 多跳执行（逐步）

第一跳：找鲁迅创作的作品；第二跳：找各作品发表于何处；第三跳：求共同 Y。

**JOIN**

```text
works = {t | (鲁迅,创作,t) in edges}
venues[w] = {t | (w,发表于,t) in edges}
```

In [8]:
for line in graph_multihop(): print(line)

鲁迅创作: 狂人日记, 呐喊
狂人日记 发表于: 文学周报社
呐喊 发表于: 文学周报社
共同发表于 文学周报社: ✓


**思考** · 为何必须约束「发表于」？

<details><summary>查看答案</summary>

否则创作边与人物边混连，语义错误。

</details>

## 4. 软路径计票

多条推理路径对同一答案投票；得分高者更可信。

In [9]:
path_ranking()

| 路径 | 得分 |
|------|------|
| 蛙→茅盾文学奖 | 3 |
| 红高粱→典藏 | 2 |
| 红高粱→电影→金熊奖 | 3 |

最高分路径: 蛙→茅盾文学奖


## 5. 硬约束 vs 软推断

**硬约束**：关系类型必须匹配模板。**软推断**：多路径聚合得分。

## 小结

多跳 = 带关系约束的图 JOIN；软路径计票用于不确定性下的排序。

对照 [ch6.html](../ch6.html) 查看路径高亮。

## 练习

1. 若去掉「发表于」约束，会多出哪些错误 Y？
2. 计票得分相同时如何打破平局？